In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Linear Models

## Data Loading and Preprocessing

In [18]:
import sys
sys.path.append("..")
from src.data import load_data, compute_returns, create_sequences

In [19]:
df = load_data()
df_returns = df.copy()
stock_cols = ['AMZN', 'DPZ', 'BTC', 'NFLX']
df_returns[stock_cols] = df_returns[stock_cols].pct_change()
df_returns = df_returns.dropna()

# Scale asset values between -1 and 1
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_returns[stock_cols])

# Split time series data into training samples
X, y = create_sequences(scaled_data, window_size=60)

# Prep data for linear models
n_samples, window, n_features = X.shape

X_lin = X.reshape(n_samples, window * n_features)

# Train/val split WITHOUT shuffling (as we're dealing with sequential time series data)
split = int(0.8 * len(X))
X_train_lin, X_val_lin = X_lin[:split], X_lin[split:]
y_train_lin, y_val_lin = y[:split], y[split:]

## Ridge regression

In [20]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

model = Ridge(alpha=0.1)
model.fit(X_train_lin, y_train_lin)

y_pred = model.predict(X_val_lin)
mse = mean_squared_error(y_val_lin, y_pred)

print("Ridge MSE:", mse)

Ridge MSE: 1.2956529344530852


## Lasso

In [21]:
from sklearn.linear_model import Lasso

model = Lasso(alpha=0.001)
model.fit(X_train_lin, y_train_lin[:, 0])

y_pred = model.predict(X_val_lin)
mse = mean_squared_error(y_val_lin[:, 0], y_pred)

print("Lasso MSE:", mse)

Lasso MSE: 1.601635806270193


## ElasticNet

In [22]:
from sklearn.linear_model import ElasticNet

model = ElasticNet(alpha=0.001, l1_ratio=0.5)
model.fit(X_train_lin, y_train_lin[:, 0])

y_pred = model.predict(X_val_lin)
mse = mean_squared_error(y_val_lin[:, 0], y_pred)
print("ElasticNet MSE:", mse)

ElasticNet MSE: 1.6092566914550026


## AR

In [23]:
from statsmodels.tsa.ar_model import AutoReg

series = df_returns["AMZN"].values
model = AutoReg(series, lags=30)
fit = model.fit()

pred = fit.predict(start=1000, end=1200)
mse = mean_squared_error(y_val_lin[:, 0], pred)
print("AR MSE:", mse)

ValueError: Found input variables with inconsistent numbers of samples: [292, 201]